# `JuMP.jl`: A Brief Initial Introduction
Here we present an initial introduction to using `JuMP.jl`. We will focus on the core modeling aspects as motivated by a simple linear programming problem. Note that this content takes inspiration from https://jump.dev/JuMP.jl/stable/tutorials/getting_started/getting_started_with_JuMP/.

## Resources
We will not be able to cover all of `JuMP.jl`'s capabilities today. Good references are:
- The tutorials, examples, manuals, and guides in `JuMP.jl`'s documentation: https://jump.dev/JuMP.jl/stable/
- The Julia optimization forum: https://discourse.julialang.org/c/domain/opt/13
- Julia Programming for Operations Research 2/e (not always up-to-date): https://www.softcover.io/read/7b8eb7d0/juliabook2/introduction

## Installation
Let's get started by installing the necessary packages:


In [1]:
import Pkg
Pkg.add(["JuMP", "HiGHS", "Ipopt", "MathOptInterface"])

    Updating registry at `C:\Users\Pulsipher\.julia\registries\General.toml`
   Resolving package versions...
   Installed JSON3 ──────────────── v1.14.2
   Installed SpecialFunctions ───── v2.5.1
   Installed Hwloc_jll ──────────── v2.12.0+0
   Installed SPRAL_jll ──────────── v2024.5.8+0
   Installed MUMPS_seq_jll ──────── v500.700.301+0
   Installed BenchmarkTools ─────── v1.6.0
   Installed MutableArithmetics ─── v1.6.4
   Installed IrrationalConstants ── v0.2.4
   Installed Bzip2_jll ──────────── v1.0.9+0
   Installed Parsers ────────────── v2.8.3
   Installed Ipopt ──────────────── v1.10.2
   Installed StaticArraysCore ───── v1.4.3
   Installed CodecBzip2 ─────────── v0.8.5
   Installed OrderedCollections ─── v1.8.0
   Installed NaNMath ────────────── v1.1.3
   Installed JLLWrappers ────────── v1.7.0
   Installed Ipopt_jll ──────────── v300.1400.1700+0
   Installed OpenBLAS32_jll ─────── v0.3.29+0
   Installed ForwardDiff ────────── v1.0.1
   Installed CommonSubexpressions ─ v0.3

Here `HiGHS` acts an appropriate LP solver and `Ipopt` acts as an appropriate NLP solver. The list of supported solvers and the problems types they can solve is provided at https://jump.dev/JuMP.jl/stable/installation/#Supported-solvers.

## Motivating Example
Consider the following linear program (LP):
$$
\begin{aligned}
& \min && 12x + 20y \\
& \;\;\text{s.t.} && 6x + 8y \geq 100 \\
&&& 7x + 12y \geq 120 \\
&&& x \geq 0 \\
&&& y \in [0, 3] \\
\end{aligned}
$$
Let's formulate this problem in `JuMP.jl` and use the HiGHS solver:

In [3]:
using JuMP, HiGHS

model = Model(HiGHS.Optimizer)

@variable(model, x >= 0)
@variable(model, 0 <= y <= 3)

@objective(model, Min, 12x + 20y)

@constraint(model, c1, 6x + 8y >= 100)
@constraint(model, c2, 7x + 12y >= 120)

print(model)

latex_formulation(model)

Min 12 x + 20 y
Subject to
 c1 : 6 x + 8 y >= 100
 c2 : 7 x + 12 y >= 120
 x >= 0
 y >= 0
 y <= 3


$$ \begin{aligned}
\min\quad & 12 x + 20 y\\
\text{Subject to} \quad & 6 x + 8 y \geq 100\\
 & 7 x + 12 y \geq 120\\
 & x \geq 0\\
 & y \geq 0\\
 & y \leq 3\\
\end{aligned} $$

That's all we have to do formulate the model and view it. Now let's optimize it!

In [5]:
optimize!(model)

@show termination_status(model)
@show primal_status(model)
@show dual_status(model)
@show objective_value(model)
@show value(x)
@show value(y)
@show shadow_price(c1)
@show shadow_price(c2);

LP   has 2 rows; 2 cols; 4 nonzeros
Coefficient ranges:
  Matrix [6e+00, 1e+01]
  Cost   [1e+01, 2e+01]
  Bound  [3e+00, 3e+00]
  RHS    [1e+02, 1e+02]
Solving LP without presolve, or with basis, or unconstrained
Model status        : Optimal
Objective value     :  2.0500000000e+02
Relative P-D gap    :  1.3864248503e-16
HiGHS run time      :          0.00
termination_status(model) = MathOptInterface.OPTIMAL
primal_status(model) = MathOptInterface.FEASIBLE_POINT
dual_status(model) = MathOptInterface.FEASIBLE_POINT
objective_value(model) = 204.99999999999997
value(x) = 15.000000000000005
value(y) = 1.249999999999996
shadow_price(c1) = -0.24999999999999922
shadow_price(c2) = -1.5000000000000007


That was pretty easy and all the results can be queried with a solver independent API. Below let's go over what we just did, step by step.

## Package Importing
To write `JuMP.jl` programs, we'll need to import `JuMP` and an appropriate solver package to solve to the model. Hence, in this case we import `JuMP` and `HiGHS`: 

In [6]:
using JuMP, HiGHS

## `JuMP.jl` Models
`JuMP` builds problems incrementally in a `Model` object. Create a model by passing an optimizer to the `Model` function:

In [7]:
model = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

Here, the convention for the optimizer input is `SolverName.Optimizer`.

## Decision Variables
`JuMP.jl`'s modeling API principally uses macros to provide an intuitive symbolic interface. For adding/creating optimization variables, we use the `@variable` macro. To define, $x \geq 0$ we write:

In [8]:
@variable(model, x >= 0)

x

To add $0 \leq y \leq 30$, we can write:

In [9]:
@variable(model, 0 ≤ y ≤ 30)

y

Notice that we used `≤` (from `\leq` and pressing [TAB]) instead of `<=` to highlight how we can use unicode characters instead if we prefer.

## Objective
The objective function is specified via `@objective`. Hence, to set $\text{min} \; 12x + 20y$ we write:

In [10]:
@objective(model, Min, 12x + 20y)

12 x + 20 y

## Constraints
Constraints are added via `@constraint`. Here, we name our constraints `c1` and `c2` for convenience in querying results later on (this is optional and the argument can be omitted if wanted).

In [11]:
@constraint(model, c1, 6x + 8y >= 100)

c1 : 6 x + 8 y >= 100

In [12]:
@constraint(model, c2, 7x + 12y >= 120)

c2 : 7 x + 12 y >= 120

## Printing the Model
Simply showing the model results in a summary of what components it has:

In [13]:
model

A JuMP Model
├ solver: HiGHS
├ objective_sense: MIN_SENSE
│ └ objective_function_type: AffExpr
├ num_variables: 2
├ num_constraints: 5
│ ├ AffExpr in MOI.GreaterThan{Float64}: 2
│ ├ VariableRef in MOI.GreaterThan{Float64}: 2
│ └ VariableRef in MOI.LessThan{Float64}: 1
└ Names registered in the model
  └ :c1, :c2, :x, :y

We have model with 2 optimization variables, an affine minimization objective, and 5 constraints of three different types. 

More conveniently we can print the model using `print`:

In [14]:
print(model)

Min 12 x + 20 y
Subject to
 c1 : 6 x + 8 y >= 100
 c2 : 7 x + 12 y >= 120
 x >= 0
 y >= 0
 y <= 30


That is certainly more human-readable. Since, we are using a Jupyter notebook, we can even print the latex formulation of our model using `latex_formulation`:

In [15]:
latex_formulation(model)

$$ \begin{aligned}
\min\quad & 12 x + 20 y\\
\text{Subject to} \quad & 6 x + 8 y \geq 100\\
 & 7 x + 12 y \geq 120\\
 & x \geq 0\\
 & y \geq 0\\
 & y \leq 30\\
\end{aligned} $$

## Optimization
Now that we have a model, let's optimize it using `optimize!`:

In [16]:
optimize!(model)

Running HiGHS 1.10.0 (git hash: fd8665394e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 2 rows; 2 cols; 4 nonzeros
Coefficient ranges:
  Matrix [6e+00, 1e+01]
  Cost   [1e+01, 2e+01]
  Bound  [3e+01, 3e+01]
  RHS    [1e+02, 1e+02]
Presolving model
2 rows, 2 cols, 4 nonzeros  0s
2 rows, 2 cols, 4 nonzeros  0s
Presolve : Reductions: rows 2(-0); columns 2(-0); elements 4(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 2(220) 0s
          2     2.0500000000e+02 Pr: 0(0) 0s
Model status        : Optimal
Simplex   iterations: 2
Objective value     :  2.0500000000e+02
Relative P-D gap    :  0.0000000000e+00
HiGHS run time      :          0.00


We will review methods later to specify solver options. One common one is `set_silent` which turns off the raw solver output:

In [17]:
set_silent(model)
optimize!(model)

## Querying Results
Our model is now optimized, so let's see what happened using `JuMP.jl`'s general purpose query API. 

We can see the final status of the solver (i.e., why it stopped) using `termination_status`:

In [18]:
termination_status(model)

OPTIMAL::TerminationStatusCode = 1

Here, it stopped because it found the optimal solution. For a list of the possible statuses see https://jump.dev/JuMP.jl/stable/moi/reference/models/#MathOptInterface.TerminationStatusCode.

We can also check whether the solver found a primal feasible point using `primal_status`:

In [19]:
primal_status(model)

FEASIBLE_POINT::ResultStatusCode = 1

It did find a feasible point. We can make the same check for the dual problem via `dual_status`:

In [20]:
dual_status(model)

FEASIBLE_POINT::ResultStatusCode = 1

We also found a dual feasible point. The list of possible statuses is provided at https://jump.dev/JuMP.jl/stable/moi/reference/models/#MathOptInterface.ResultStatusCode.

To keep things simple, we can even just use `is_solved_and_optimal`:

In [22]:
is_solved_and_feasible(model)

true

Now we know that we have an optimal solution with feasible primal and dual solutions that we can interrogate.

Query the objective value via `objective_value`:

In [23]:
objective_value(model)

205.0

Now find the variable values using `value`:

In [24]:
@show value(x)
@show value(y);

value(x) = 15.000000000000004
value(y) = 1.2499999999999976


Finally, we can learn about the dual solution using `shadow_price`:

In [25]:
@show value(c1)
@show value(c2);

value(c1) = 100.0
value(c2) = 120.0


We could have instead used `dual` to get the raw dual values, but `shadow_price` corrects the signs in accordance with the objective sense to have a consistent interpretation.

## Exercise: Simple QP Model
**Problem**
- Solve the following model using `JuMP.jl` using the `HiGHS` solver

$$
\begin{aligned}
& \min && 3x^2 + 2y^2 - 4x \\
& \;\;\text{s.t.} && 6x - 8y \geq 100 \\
&&& x + 12y = 120 \\
&&& x \geq 0 \\
&&& y \in [0, 3] \\
\end{aligned}
$$

In [ ]:
# PUT CODE HERE


## Working with Solvers
We often not only want to specify a solver, but also want to set some attributes as well. Here, the attributes are solver specific and can be found by checking the documentation associated with each solver. We can also specify/modify attributes using `set_attribute`:

In [30]:
model = Model(HiGHS.Optimizer)
set_attribute(model, "output_flag", false)
set_attribute(model, "presolve", "on")

For convenience, `JuMP.jl` provides a few solver-agnostic methods for setting common attributes such as turning the output off and setting a time limit:

In [27]:
model = Model(HiGHS.Optimizer)
set_silent(model) # turn the output printing off
set_time_limit_sec(model, 60.0) # set a time limit

## Variables
Let's take a deeper dive into more of the things we can do with `@variable`.

### Containers and Sets
We have already seen how to add individual scalar variables, now let's see how to add multiple variables at once.

`JuMP.jl` uses 3 data structures to store variable collections:
- `Array`s: The native Julia arrays
- `DenseAxisArray`s: Dense arrays with arbitrary indices
- `SparseAxisArray`s: Sparse arrays with arbitrary indices

Arrays are created using integer indices of the form `1:n`. For instance, the matrix:

In [31]:
model = Model()
@variable(model, a[1:2, 1:4])

2×4 Matrix{VariableRef}:
 a[1,1]  a[1,2]  a[1,3]  a[1,4]
 a[2,1]  a[2,2]  a[2,3]  a[2,4]

This creates a 2 x 4 matrix of variables that is stored to `a` which we can index and use in defining our problem.

We can also create an n-dimensional vector variable $x \in \mathbb{R}^n$ with upper and lower bounds:

In [32]:
n = 5
l = [1, 2, 3, 4, 5]
u = [10, 11, 12, 13, 14]

@variable(model, l[i] <= x[i = 1:n] <= u[i])

5-element Vector{VariableRef}:
 x[1]
 x[2]
 x[3]
 x[4]
 x[5]

Notice we declare an index `i` to help us define the appropriate values. 

We can use other index forms that don't conform to `1:n` and make `DenseAxisArray`s:

In [33]:
@variable(model, z[i = 2:3, j = 1:2:3] >= i + 2j)

2-dimensional DenseAxisArray{VariableRef,2,...} with index sets:
    Dimension 1, 2:3
    Dimension 2, 1:2:3
And data, a 2×2 Matrix{VariableRef}:
 z[2,1]  z[2,3]
 z[3,1]  z[3,3]

We don't even have to use integers:

In [34]:
@variable(model, w[["red", "blue"], 1:5] <= 1)

2-dimensional DenseAxisArray{VariableRef,2,...} with index sets:
    Dimension 1, ["red", "blue"]
    Dimension 2, Base.OneTo(5)
And data, a 2×5 Matrix{VariableRef}:
 w[red,1]   w[red,2]   w[red,3]   w[red,4]   w[red,5]
 w[blue,1]  w[blue,2]  w[blue,3]  w[blue,4]  w[blue,5]

For indices that do not form a rectangular set, a `SparseAxisArray` is formed:

In [35]:
@variable(model, u[i = 1:2, j = i:3])

JuMP.Containers.SparseAxisArray{VariableRef, 2, Tuple{Int64, Int64}} with 5 entries:
  [1, 1]  =  u[1,1]
  [1, 2]  =  u[1,2]
  [1, 3]  =  u[1,3]
  [2, 2]  =  u[2,2]
  [2, 3]  =  u[2,3]

We can even add a conditional statement after a `;` when defining indices:

In [36]:
@variable(model, v[i = 1:3, j = 1:4; i + j <= 4])

JuMP.Containers.SparseAxisArray{VariableRef, 2, Tuple{Int64, Int64}} with 6 entries:
  [1, 1]  =  v[1,1]
  [1, 2]  =  v[1,2]
  [1, 3]  =  v[1,3]
  [2, 1]  =  v[2,1]
  [2, 2]  =  v[2,2]
  [3, 1]  =  v[3,1]

### Integrality
To specify integer variables, we need only add the `Int` argument:

In [37]:
@variable(model, integer_x, Int)

integer_x

Similarly, we create binary variables via the `Bin` argument:

In [ ]:
@variable(model, binary_x, Bin)

### Exercise: Nodal Variables
**Problem**
- Create a variable named `xp`
- `xp` should be integer valued between 0 and 3
- `xp` should be indexed over each arc `(i, j)` in `arcs`

In [ ]:
arcs = [(1, 2), (1, 3), (3, 2), (2, 4)]

# PUT CODE HERE


### Other Options
There are a variety of other things we can do with variables. We can create a fixed variable:

In [ ]:
@variable(model, x_fixed == 42)

We can specify the initial guess to pass on to the solver via `start`:

In [ ]:
@variable(model, q, start = 2)

### Modify Variables
There are a variety of ways to change variables after they are created. Some common methods include:
- `set_lower_bound`
- `set_upper_bound`
- `fix`
- `set_start_value`
- `set_binary`
- `set_integer`
- `delete`

For example:

In [ ]:
set_upper_bound(x, 10)
set_integer(x)
delete(model, x)

There are many more things we can do, see https://jump.dev/JuMP.jl/stable/manual/variables/ to learn more. 